# 01 First EDA (preliminary)

**Owner:** ER · **Date:** 2026-11-04 (first version 2026-11-03 with one week; this version uses the full build)

**Purpose:** first descriptive look at the data for the concept presentation (issue #12). No tests, no models, no controls.

**Input:** table `analysis_base` in the project database (built with `python -m src.build_db --study-period`), **winter half-year (Nov–Mar) 2023/24, 2024/25, 2025/26**, canton of Zurich.

**Output:** tables in this notebook and three figures in `presentation/figures/` (all marked *preliminary*).

> Everything here is **preliminary and descriptive**. All arrivals in the same hour share the same weather, so the number of distinct weather hours (`hours`) matters more than the number of arrivals (`n`).

Setup: project paths and database helpers from `src/` (same code for SQLite and PostgreSQL).

In [ ]:
import sys; sys.path.append("..")
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.paths import RAW_DIR, PROCESSED_DIR, ROOT
from src.db import query, write_table, list_tables, backend

FIG_DIR = ROOT / "presentation" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Fixed colours for the whole project: regional = blue, long-distance = orange
# (checked for colour-vision deficiency with a palette validator)
COLORS = {"regional": "#2a78d6", "long_distance": "#eb6834"}
LABELS = {"regional": "Regional (S-Bahn)", "long_distance": "Long-distance (IC, IR, RE, EC ...)"}
SOURCE = ("Source: opentransportdata.swiss (Ist-Daten), MeteoSwiss; own database. "
          "Winter half-year (Nov-Mar) 2023/24-2025/26, canton ZH.")

# Recessive axes and grid for slide figures
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e6", "axes.axisbelow": True, "font.size": 11})
print("database backend:", backend())

## 1. Load data

What is in the database (`list_tables()` is also the basis for the slide *data structure*).

In [ ]:
list_tables()

The analysis table has about 10 million rows, so we never load all of it at once: aggregations run **in SQL** (`GROUP BY`), and only a 1 % random sample is loaded for the structure overview and the distribution plot. `RANDOM()` works on SQLite and PostgreSQL.

In [ ]:
n_total = query("SELECT COUNT(*) AS n FROM analysis_base")["n"][0]
sample = query("SELECT * FROM analysis_base WHERE ABS(RANDOM()) % 100 = 0")
print(f"{n_total:,} arrivals in analysis_base; sample: {len(sample):,} rows, {sample.shape[1]} columns")
sample.head()

## 2. Data structure

Columns, data types and missing values (from the 1 % sample). Missing `entry_delay_min` is expected: it is NULL when a trip starts inside the canton.

In [ ]:
structure = pd.DataFrame({
    "dtype": sample.dtypes.astype(str),
    "missing_pct": (100 * sample.isna().mean()).round(1),
    "example": sample.iloc[0],
})
structure

Size of the groups we compare (traffic type × weather), counted **in SQL** over all rows. `hours` = number of distinct weather hours behind each group.

In [ ]:
groups = query('''
    SELECT traffic_type, precip_cat,
           COUNT(*)                         AS arrivals,
           SUM(delayed)                     AS delayed,
           ROUND(100.0 * AVG(delayed), 1)   AS delay_rate_pct,
           COUNT(DISTINCT weather_hour_utc) AS hours
    FROM analysis_base
    GROUP BY traffic_type, precip_cat
    ORDER BY traffic_type, precip_cat
''')
groups

## 3. Delay rate per traffic type × dry / rain / snow (slide 10)

Delayed = arrival ≥ 3 min late (SBB definition). Label = rate, number of arrivals (n) and distinct weather hours (h).

In [ ]:
cats = ["dry", "rain", "snow"]
fig, ax = plt.subplots(figsize=(9.5, 5.2))
width = 0.38
for i, tt in enumerate(["regional", "long_distance"]):
    g = groups[groups.traffic_type == tt].set_index("precip_cat").reindex(cats)
    # bar positions: two bars per weather class with a small gap between them
    x = [c + (i - 0.5) * (width + 0.02) for c in range(len(cats))]
    ax.bar(x, g.delay_rate_pct, width=width, color=COLORS[tt], label=LABELS[tt])
    for xi, rate, n, h in zip(x, g.delay_rate_pct, g.arrivals, g.hours):
        ax.text(xi, rate + 0.4, f"{rate:.1f} %\nn={n:,}\nh={h:,}", ha="center", va="bottom", fontsize=8.5, color="#333333")
ax.set_xticks(range(len(cats)), ["Dry", "Rain", "Snow"])
ax.set_ylabel("Arrivals >= 3 min late (%)")
ax.set_ylim(0, groups.delay_rate_pct.max() * 1.45)
ax.legend(frameon=False, loc="upper left")
ax.set_title("PRELIMINARY: trains are late more often in rain and much more often in snow",
             loc="left", fontsize=13)
fig.text(0.01, -0.04, SOURCE + "\nWeather = last full hour before the scheduled arrival; n = arrivals, "
         "h = distinct weather hours. Descriptive, no controls, no test.", fontsize=8, color="#555555")
fig.tight_layout()
fig.savefig(FIG_DIR / "prelim_delay_rate_by_weather.png", dpi=200, bbox_inches="tight")
plt.show()

Main comparison decided for the project: precipitation (rain + snow) vs. dry, per traffic type. Absolute difference in percentage points and, as robustness check, the ratio (descriptive only; the tests belong to issue #20).

In [ ]:
main = query('''
    SELECT traffic_type, precip_any, COUNT(*) AS arrivals, 100.0 * AVG(delayed) AS delay_rate_pct
    FROM analysis_base GROUP BY traffic_type, precip_any
''').pivot(index="traffic_type", columns="precip_any", values="delay_rate_pct")
main.columns = ["dry_pct", "precipitation_pct"]
main["difference_pp"] = main.precipitation_pct - main.dry_pct
main["ratio"] = main.precipitation_pct / main.dry_pct
main.round(2)

## 4. Distribution of the arrival delays

From the 1 % sample. Values outside −2 … 15 min are grouped at the edges.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
bins = [x / 2 for x in range(-4, 31)]  # half-minute bins from -2 to 15 min
for ax, tt in zip(axes, ["regional", "long_distance"]):
    d = sample.loc[sample.traffic_type == tt, "arr_delay_min"].clip(-2, 15)
    # weights turn counts into percent of arrivals, so both panels are comparable
    ax.hist(d, bins=bins, color=COLORS[tt], weights=[100 / len(d)] * len(d), edgecolor="white", linewidth=0.5)
    ax.axvline(3, color="#333333", lw=1.2, ls="--")
    ax.set_title(f"{LABELS[tt]}\nsample n={len(d):,}, median {d.median():.1f} min", fontsize=11, loc="left")
    ax.set_xlabel("Arrival delay (min), clipped to -2 ... 15")
for ax in axes:
    ax.text(3.2, ax.get_ylim()[1] * 0.92, "3 min threshold", fontsize=9)
axes[0].set_ylabel("Share of arrivals (%)")
fig.suptitle("PRELIMINARY: most arrivals are about 1 min late; only the tail passes 3 min",
             x=0.01, ha="left", fontsize=13)
fig.text(0.01, -0.03, SOURCE + " 1 % random sample.", fontsize=8, color="#555555")
fig.tight_layout()
fig.savefig(FIG_DIR / "prelim_delay_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Delay rate per month

Month-to-month differences, aggregated in SQL. `SUBSTR` works on SQLite and PostgreSQL.

In [ ]:
monthly = query('''
    SELECT SUBSTR(local_date, 1, 7) AS month, traffic_type, COUNT(*) AS arrivals,
           ROUND(100.0 * AVG(delayed), 1) AS delay_rate_pct,
           ROUND(100.0 * AVG(precip_any), 1) AS share_with_precip_pct
    FROM analysis_base
    GROUP BY SUBSTR(local_date, 1, 7), traffic_type ORDER BY month
''')
# keep the study months only (a few night arrivals fall on 01.04.)
monthly = monthly[monthly.month.str[5:].isin(["11", "12", "01", "02", "03"])]
fig, ax = plt.subplots(figsize=(11, 4.5))
for tt in ["regional", "long_distance"]:
    g = monthly[monthly.traffic_type == tt]
    ax.plot(g.month, g.delay_rate_pct, marker="o", ms=8, lw=2, color=COLORS[tt], label=LABELS[tt])
ax.set_ylabel("Arrivals >= 3 min late (%)")
ax.set_ylim(0, monthly.delay_rate_pct.max() * 1.15)
ax.legend(frameon=False)
ax.tick_params(axis="x", rotation=45)
ax.set_title("PRELIMINARY: delay rates differ between months and winters", loc="left", fontsize=13)
fig.text(0.01, -0.12, SOURCE, fontsize=8, color="#555555")
fig.tight_layout()
fig.savefig(FIG_DIR / "prelim_delay_rate_by_month.png", dpi=200, bbox_inches="tight")
plt.show()
monthly.pivot(index="month", columns="traffic_type", values=["delay_rate_pct", "share_with_precip_pct"])

## Findings

*(for the slides, all preliminary and descriptive, winter half-year 2023/24–2025/26)*

- About 10 million train arrivals at 157 stops in canton ZH; every arrival has a precipitation value and a calendar day.
- In both traffic types, the share of late arrivals is higher in rain hours than in dry hours, and much higher in snow hours (see the table in section 3 for the exact values).
- Long-distance trains have the higher delay rate in every weather class.
- Snow rests on few weather hours (column `hours`), so the snow values are uncertain.
- The absolute difference (percentage points) and the ratio can point in different directions between the traffic types (section 3, table `main`). This is exactly why the primary effect measure was fixed in advance (absolute, decided 2026-11-03). No conclusion here: tests and controls belong to issue #20.

## Open questions

- Arrivals in the same hour are not independent → effect sizes with confidence intervals at hour or day level (issue #20).
- Month-to-month differences (section 5) need the controls `month` and `winter`.
- Four days (28.–31.01.2025) are missing: the archive uses Deflate64 compression for them, which Python's `zipfile` cannot read.

### Jupyter notebook --footer info--

In [ ]:
import os, platform
from platform import python_version
from datetime import datetime
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')